In [1]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
def restart_router():
    print("Start restarting process")
    try:
        chrome_options = Options()
        chrome_options.add_argument("--no-sandbox")
        # chrome_options.add_argument("--headless")
        chrome_options.add_argument("--start-maximized")
        chrome_options.add_argument("--window-size=1920,1080")
        chrome_options.add_argument("--ignore-certificate-errors")
       
        driver = webdriver.Chrome(options=chrome_options)
        driver.get("http://192.168.1.1")
        time.sleep(10)
        driver.find_element(By.ID,"login-txt-pwd").send_keys("")
        driver.find_element(By.ID,"login-btn-logIn").click()
        time.sleep(10)
        driver.find_element(By.XPATH,"/html/body/div[1]/div[2]/div/div/ul/li[6]/a").click()
        time.sleep(10)
        driver.find_element(By.XPATH,'//*[@id="restart"]').click()
        time.sleep(6)
        driver.find_element(By.ID,"restart-btn-reconnect_dsl").click()
        time.sleep(7)
        driver.find_element(By.ID,"restart-dsl-btn-apply").click()
        
        print("Restart successfully!")
        driver.quit()
        return True
    except:
        driver.quit()
        return False

In [2]:
ITALY_DATA = {"countryCode": "gbr","missionCode": "ita","vacCode": "ILON","visaCategoryCode": "TBE"}
    
NLD_DATA = {"countryCode":"gbr","missionCode":"nld","vacCode":"NAKN","visaCategoryCode":"SV"}
from email import message_from_bytes
import email.utils as email_utility
import imaplib
from datetime import datetime, timedelta
import traceback
def get_otp():
        """Function uses the IMAP library to pull out
        the latest otp sent to our email

        Returns:
            jiea sxtx iyxm brhu
            None|Int: _description_
        """
        # Set up your email and password
        username = "ali.hammad@thesemantics.co"
        app_password = ";wZ]h&wYXPw~"  # Use App Password for Gmail

        try:
            mail = imaplib.IMAP4_SSL("thesemantics.co")
            mail.login(username, app_password)

            # Select the mailbox you want to check (e.g., inbox)
            mail.select("inbox")

            # Search Subject
            search_subject = "One Time Password"

            # Search for emails since today
            _, messages = mail.search(None, f'SUBJECT "{search_subject}"')

            # Convert the messages to a list of email IDs
            email_ids = messages[0].split()
            # Fetch the most recent email (last in the list)
            if email_ids:
                latest_email_id = email_ids[-1]
                _, msg_data = mail.fetch(latest_email_id, "(RFC822)")
                raw_email = msg_data[0][1]
                msg = message_from_bytes(raw_email)
                date_tuple = email_utility.parsedate_tz(msg["Date"])
                if date_tuple:
                    email_date = datetime.fromtimestamp(email_utility.mktime_tz(date_tuple))

                    # Get the current time and calculate the time 5 minutes ago
                    current_time = datetime.now()
                    time_5_minutes_ago = current_time - timedelta(minutes=5)

                    # Check if the email was received in the last 5 minutes
                    if email_date > time_5_minutes_ago:
                        while msg.is_multipart():
                            msg = msg.get_payload(0)
                        body = msg.get_payload(decode=True).decode()
                        # Extract OTP from the email body using a regex pattern (adjust the pattern as needed)
                        import re

                        otp = re.search(r"\d{6}", body)  # Example pattern for a 6-digit OTP

                        if otp:
                            mail.logout()
                            print("OTP found")
                            return otp.group()
                        else:
                            print("OTP not found in the email.")
                    else:
                        print("No recent emails found within the last 5 minutes.")
                else:
                    print("Could not parse the email date.")
            else:
                print("No emails found with the specified subject.")
            # Logout from the email account
            mail.logout()
            return None
        except Exception as e:
            print(traceback.format_exc())

In [ ]:
import json
import time
from seleniumbase import SB
import traceback
from datetime import datetime
from twilio.rest import Client
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By

country_data = {
    "nld": {
        "countryCode": "gbr",
        "missionCode": "nld",
        "vacCode": [("London", "NTBM")],
        "visaCategoryCode": "TA",
    },
    "prt": {
        "countryCode": "gbr",
        "missionCode": "prt",
        "vacCode": ["PRT-LON"],
        "visaCategoryCode": "TV",
    },
}

class VfsScrapper:
    def __init__(self, country, email, password) -> None:
        self.email = email
        self.password = password
        self.country = country
        self.auth_token = None
        self.vaccode_counter = 0
        self.city_count = None

    def get_configuration(self, sb, country, mission, vac_code):
        """Fetch the configuration data."""
        url = f"https://lift-api.vfsglobal.com/configuration/fields/{mission}/{country}/{vac_code}"
        headers = {
            "Content-Type": "application/json",
            "Authorize": self.auth_token,
        }
        script = (
            f"let [resolve] = arguments; "
            f"fetch('{url}', {{"
            f"  method: 'GET',"
            f"  headers: {{ 'Content-Type': 'application/json', 'Authorize': '{self.auth_token}' }}"
            f"}})"
            f".then(response => response.json())"
            f".then(result => resolve(result))"
            f".catch(error => resolve(error.toString()));"
        )
        response = sb.driver.execute_async_script(script)
        if isinstance(response, str):
            print("Error fetching configuration:", response)
            raise Exception("Failed to fetch configuration.")
        self.configuration_payload = response
        print("Configuration fetched:", self.configuration_payload)

    def _get_post_call(self, country_data, city_index):
        payload = {
            "countryCode": country_data["countryCode"],
            "missionCode": country_data["missionCode"],
            "vacCode": country_data["vacCode"][city_index][1],
            "visaCategoryCode": country_data["visaCategoryCode"],
            "roleName": "Individual",
            "loginUser": self.email,
            "payCode": ""
        }

        headers = {
            "Authorize": self.auth_token,
            "Content-Type": "application/json"
        }

        script = (
            "let [resolve] = arguments; "
            "const myHeaders = new Headers(); "
            "myHeaders.append('Authorize', '" + headers["Authorize"] + "'); "
            "myHeaders.append('Content-Type', 'application/json'); "
            "const raw = JSON.stringify(" + json.dumps(payload) + "); "
            "const requestOptions = { method: 'POST', headers: myHeaders, body: raw, redirect: 'follow' }; "
            "fetch('https://lift-api.vfsglobal.com/appointment/CheckIsSlotAvailable', requestOptions) "
            ".then(response => response.json()) "
            ".then(result => resolve(result)) "
            ".catch(error => resolve(error.toString()));"
        )

        return script
    def _open_the_turnstile_page(self, sb):
        sb.driver.uc_open_with_reconnect(
            f"https://visa.vfsglobal.com/gbr/en/{self.country}/login",
            reconnect_time=4.7,
        )

    def _is_successfull(self, sb):
        return True
        iframes = sb.find_elements("iframe") 
        for iframe in iframes:
            # Switch to the iframe
            sb.switch_to_frame(iframe)

            # Example: Do something inside the iframe
            elements = sb.find_elements("*")  # This selects all elements

        # Collect IDs of all elements
            ids = [element.get_attribute("id") for element in elements if element.get_attribute("id")]

        # Output the collected IDs
            print("IDs of all elements inside the iframe:", ids)
            if sb.is_text_visible("Success!"):
                print("success")
                return True
           
            # Switch back to the default content after each iframe
            sb.switch_to_default_content()
        return False

    def _click_turnstile_and_verify(self, sb):
        sb.save_screenshot("test")
        sb.uc_gui_click_captcha()
        if self._is_successfull(sb):
            return True
        else:
            try:
                sb.uc_gui_click_captcha()
                if self._is_successfull(sb):
                    return True
                raise Exception
            except Exception as e:
                sb.execute_script(
                    f"window.open('https://visa.vfsglobal.com/gbr/en/{self.country}/login')"
                )
                sb.sleep(15)
                sb.driver.close()
                sb.switch_to_window(0)
                if self._is_successfull(sb):
                    return True
                else:
                    sb.uc_gui_click_captcha()
                    if self._is_successfull(sb):
                        return True
                    return False

    def check_slot_avalaible(self, sb):
        message = ""
        continue_looking = True
        self.city_count = len(country_data[self.country]["vacCode"])
        city_index = self.vaccode_counter % self.city_count
        self.vaccode_counter += 1

        script = self._get_post_call(country_data[self.country], city_index=city_index)
        post_request = sb.driver.execute_async_script(script)

        city = country_data[self.country]["vacCode"][city_index][0]

        if isinstance(post_request, str):
            print("Error in response: ", post_request)
            raise Exception("Invalid response format")

        if post_request.get("status", None) == 401:
            print(self.country)
            print("401 Status Code")
            raise Exception("Need to ReAuthorize")

        if post_request.get("earliestDate", None):
            date_str = post_request.get("earliestDate")
            date_format = "%m/%d/%Y %H:%M:%S"
            date_obj = datetime.strptime(date_str, date_format)
            message = f"{self.country} {city} earliestDate {post_request['earliestDate']}"
            print(message)
            continue_looking = False
            self.send_sms(message)

        print(f"Request made for {city}")
        return post_request, continue_looking, message

    def send_sms(self, message):
        account_sid = "AC0ce99e0bd3abd7748fa67cd01607c54e"
        auth_token = "83983a62faebdfc2cabb089cbf81d6da"
        client = Client(account_sid, auth_token)

        client.messages.create(
            body=f"VFS Appointments {message} available",
            from_="+447700101592",
            to="+447724267222",
        )

    def fill_credentials(self, sb_object) -> None:
        sb_object.type("#email", self.email)
        sb_object.driver.uc_click("#mat-input-4")

        key_sequence = ["{shift}", "F", "{shift}", "a", "s", "t", "i", "a", "n", "{numeric}", "1", "5", "!"]

        for key in key_sequence:
            if key == "{shift}":
                sb_object.click("button[name='{shift}']")
            elif key == "{numeric}":
                sb_object.click("button[name='{numeric}']")
            else:
                sb_object.click(f"button[name='{key}']")
            time.sleep(0.5)

    def fill_otp(self, sb_object, sb_otp) -> None:
        sb_object.driver.uc_click("#mat-input-5")
        otp = str(sb_otp)
        for char in otp:
            sb_object.click(f"button[name='{char}']")
            time.sleep(0.5)

    def get_auth_token(self):
        auth_token = None
        with SB(
            uc=True,
            user_data_dir="C:/Users/Dev/Documents/Projects/VFS_SCRAPER/.seleniumtests"
        ) as sb:
            try:
                self._open_the_turnstile_page(sb)
                verify_human = self._click_turnstile_and_verify(sb)
                if verify_human:
                    self.fill_credentials(sb_object=sb)
                    sb.driver.uc_click("button.mat-btn-lg")
                    sb.sleep(10)
                    print(get_otp())
                    self.fill_otp(sb_otp=get_otp(), sb_object=sb)
                    sb.driver.uc_click("button.mat-btn-lg")
                    sb.sleep(10)
                    auth_token = sb.driver.execute_script(
                        "return sessionStorage.getItem('JWT')"
                    )
            except Exception as e:
                print(e)
                sb.driver.quit()
        return auth_token

    def start_script(self):
        consecutive_rejections = 0
        continue_looking = True
        message = ""

        for i in range(1):
            print("Opening new browser")
            self.auth_token = self.get_auth_token()
            if self.auth_token is not None:
                consecutive_rejections = 0
                with SB(uc=True, mobile=True, headless2=True) as sb:
                    sb.open(f"https://visa.vfsglobal.com/gbr/en/{self.country}/login")
                    city = country_data[self.country]["vacCode"][0][1]
                    self.get_configuration(sb, "gbr", "nld", city)
                    while continue_looking:
                        try:
                            post_request, continue_looking, t_message = (
                                self.check_slot_avalaible(sb)
                            )
                            message += t_message
                            print(post_request)
                            time.sleep(240 // self.city_count)
                        except TimeoutException as e:
                            print(f"Exception when doing post request: {e}")
                            sb.driver.close()
                            sb.driver.quit()
                            break
                            time.sleep(60)
                            continue
                        except Exception as e:
                            print(traceback.format_exc())
                            sb.driver.quit()
                            break
            else:
                print("Consecutive Failure")
                consecutive_rejections += 1
            print("breaking")
            break

import concurrent.futures

futures = []
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    for country, email, password in [
        ("nld", "ali.hammad@thesemantics.co", "Fastian15!")
    ]:
        vfs_scrapper = VfsScrapper(country, email, password)
        futures.append(executor.submit(vfs_scrapper.start_script))
for future in futures:
    print(future.result())


Opening new browser
OTP found
451685
OTP found
Error in response:  Invalid Request
Traceback (most recent call last):
  File "C:\Users\Dev\AppData\Local\Temp\ipykernel_17100\3834906156.py", line 225, in start_script
    self.check_slot_avalaible(sb)
  File "C:\Users\Dev\AppData\Local\Temp\ipykernel_17100\3834906156.py", line 133, in check_slot_avalaible
    raise Exception("Invalid response format")
Exception: Invalid response format

breaking
None
